# ISLES 2022 UNet


In [4]:
from IPython.display import clear_output
# Need to do this every time I spin the notebook up
!pip install -qr requirements.txt
!python -m pip install -q --upgrade torch torchvision
clear_output()

# Imports

In [3]:
import os
import glob
from pathlib import Path

import random
from math import floor, ceil
import numpy as np
import torch
from torch.nn import functional as F
import matplotlib.pyplot as plt

from monai.utils import first, set_determinism
from monai.transforms import (
    AsDiscrete,
    AsDiscreted,
    EnsureChannelFirstd,
    Compose,
    CropForegroundd,
    LoadImaged,
    Orientationd,
    RandCropByPosNegLabeld,
    ResizeWithPadOrCropd,
    RandCropByLabelClassesd,
    Rotated,
    SaveImaged,
    ScaleIntensityRanged,
    Spacingd,
    EnsureTyped,
    EnsureType,
    Invertd,
    Rotate90d,
    SplitDimd,
)
# For some reason doesn't want to import
# from SplitDimD import SplitDimd

from monai.handlers.utils import from_engine
from monai.networks.nets import UNet
from monai.networks.layers import Norm
from monai.metrics import DiceMetric
from monai.losses import DiceLoss, DiceCELoss
from monai.inferers import SlidingWindowInferer
from monai.data import CacheDataset, DataLoader, Dataset, decollate_batch
from monai.config import print_config
from monai.apps import download_and_extract

ImportError: cannot import name 'SplitDimd'

# Class to create a batch of 2D images
batch_size is the number of slices to send together

In [ ]:
class slice_generator():
    def __init__(self, data, target, batch_size, downscale=True):
        self.store = {'image':[], 'mask':[]}
        self.images_stroke = []
        self.images_background = []
        self.masks_stroke = []
        self.masks_background = []
        self.count = 0
        
        
        # makes the first dimension of the image the axial dimension (for visualisation purposes)
        data = torch.movedim(data, 4, 2)
        target = torch.movedim(target, 4, 2)
        
        def downscale_tensor(tensor, h, w):
            # size = size you want tensor to be resized too
            from torchvision import transforms as transforms
            transform = transforms.Resize((h,w))
            return transform(tensor)

        for i, batch in enumerate(data):
            for j, channel in enumerate(batch):  
                for k, image in enumerate(channel):              
                    # add a channel
                    image = image[None, :]
                    mask = target[i][j][k]
                    # add a channel
                    mask = mask[None, :]
                    
                    if downscale:
                        image = downscale_tensor(image, 232, 188)
                        mask = downscale_tensor(mask, 232, 188)
                    
                    ''' if the mask has a stroke - store it in the first list '''
                    if 1 in mask:
                        self.images_stroke.append(image)
                        self.masks_stroke.append(mask)
                    else:
                        self.images_background.append(image)
                        self.masks_background.append(mask)
        
        ''' randomise the order to aid with learning '''
        pairs = list(zip(self.images_stroke, self.masks_stroke))
        random.shuffle(pairs)
        self.images_stroke, self.masks_stroke = zip(*pairs)
        pairs = list(zip(self.images_background, self.masks_background))
        random.shuffle(pairs)
        self.images_background, self.masks_background = zip(*pairs)
        
        assert(len(self.images_stroke) > 0)
        assert(len(self.images_background) > 0)
                
        num_of_batches = floor((len(self.images_stroke) + len(self.images_background)) / batch_size)
        num_of_stroke = floor(len(self.images_stroke) / num_of_batches)
        num_of_background = floor(len(self.images_background) / num_of_batches)
        
        # print(num_of_batches)
        # print(num_of_stroke)
        # print(num_of_background)
        # print()
        # print(len(self.images_stroke))
        # print(len(self.images_background))
        # print(len(self.images_background + self.images_stroke))
                
        stroke_count = 0
        background_count = 0
        
        for _ in range(num_of_batches):
            # ensuring we get all the stroke masks within the dataset
            if (i >= (num_of_batches-1)):
                num_of_stroke = len(self.images_stroke) - stroke_count
                num_of_background = len(self.images_background) - background_count
                if num_of_stroke + num_of_background > batch_size:
                    num_of_background = batch_size - num_of_stroke

            images = []
            masks = []
            for _ in range(num_of_stroke):
                images.append(self.images_stroke[stroke_count])
                masks.append(self.masks_stroke[stroke_count])
                stroke_count += 1
            for _ in range(num_of_background):
                # print(background_count)
                images.append(self.images_background[background_count])
                masks.append(self.masks_background[background_count])
                background_count += 1
            tmp_imgs = torch.stack(images)
            tmp_msks = torch.stack(masks)
            self.store['image'].append(tmp_imgs)
            self.store['mask'].append(tmp_msks)
            images.clear()
            masks.clear()
            
        
        ''' make sure we have the same amount of masks and images '''
        assert len(self.store['image']) == len(self.store['mask'])
        self.length = len(self.store['image'])    
        
    def __iter__(self):
        self.count = 0
        return self
        
    def __next__(self):
        if self.count + 1 > self.length:
            raise StopIteration
        # provides a 2d image and its respective 2d mask ... (b,c,h,w) shape
        image = self.store['image'][self.count]
        mask = self.store['mask'][self.count]
        self.count += 1
        return image, mask

# If possible, change Device to GPU/Cuda


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print()

#Additional Info when using cuda
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print(torch.cuda.get_device_properties(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

# Load Data

In [ ]:
def get_paths(mask):
    data_dir = os.path.join(os.getcwd(), 'data/train')
    tmp = []
    for path in Path(data_dir).rglob(mask):
        tmp.append(path.resolve())
    return tmp

In [ ]:
training_dir = os.path.join(os.getcwd(), 'metrics')

image_paths = get_paths('*T1w*')
label_paths = get_paths('*mask*')

# To ensure I am not pulling in the training set with no mask/labels
assert(len(image_paths) == len(label_paths))
data_length = len(image_paths)

In [ ]:
data_dicts = [
    {"image": image_name, "label": label_name}
    for image_name, label_name in zip(image_paths, label_paths)
]

# Because, why not? 
assert(len(data_dicts) == data_length)

In [ ]:
from torch.utils.data import random_split

# Create train and validation sets
data_len = len(data_dicts)
val_len = int(data_len / 10)
train_len = data_len - val_len
train_files, val_files = random_split(data_dicts, [train_len, val_len])

In [ ]:
train_set = []
val_set = []
for i in range(80):
    train_set.append(train_files[i])

for i in range(8):
    val_set.append(val_files[i])

if len(train_files) != 100:
    train_files = train_set
    val_files = val_set
    
print(len(train_set))
print(len(val_set))

# Create Transforms using Monai

In [ ]:
train_transforms = Compose(
    [
        LoadImaged(keys=["image", "label"], reader='NibabelReader'), # Load image file or files from provided path based on reader.
        EnsureChannelFirstd(keys=["image", "label"]), #adds a channel dimension if the data doesn't have one ... torch.Size([1, ...]) = torch.Size([1, 1, ...
        Orientationd(keys=["image", "label"], axcodes="LPS"),
        Rotate90d(keys=["image", "label"], k=1, spatial_axes=(0,2)), # rotate data so it looks like it should do? ... doesn't feel right when viewing otherwise
        ScaleIntensityRanged(
            keys=["image"], a_min=0.0, a_max=302.0,
            b_min=0.0, b_max=1.0, clip=True,
        ),
        SplitDimd(keys=["image", "label"],dim=2),
        EnsureTyped(keys=["image", "label"], data_type='tensor'), # converts the data to a pytorch tensor
    ]
)
val_transforms = Compose(
    [
        LoadImaged(keys=["image", "label"], reader='NibabelReader'),
        EnsureChannelFirstd(keys=["image", "label"]),
        Orientationd(keys=["image", "label"], axcodes="LPS"),
        Rotate90d(keys=["image", "label"], k=1, spatial_axes=(0,2)), # rotate data so it looks like it should do? ... doesn't feel right when viewing otherwise
        ScaleIntensityRanged(
            keys=["image"], a_min=0.0, a_max=302.0,
            b_min=0.0, b_max=1.0, clip=True,
        ),
        EnsureTyped(keys=["image", "label"], data_type='tensor'),
    ]
)

# Check data loader has loaded correctly

In [ ]:
check_ds = Dataset(data=val_files, transform=val_transforms)
check_loader = DataLoader(check_ds, batch_size=1)
check_data = first(check_loader)
image, label = (check_data["image"][0][0], check_data["label"][0][0])
print(f"image shape: {image.shape}, label shape: {label.shape}")

In [ ]:
check_ds = Dataset(data=train_files, transform=train_transforms)
check_loader = DataLoader(check_ds, batch_size=1)
check_data = first(check_loader)
image, label = (check_data["image"][0][0], check_data["label"][0][0])
print(f"image shape: {image.shape}, label shape: {label.shape}")

# Visualise

In [ ]:
# Does the data look right?
plt.figure("check", (12, 6))
plt.subplot(1, 2, 1)
plt.title("image")
plt.imshow(image[:,:,60], cmap="gray")
plt.subplot(1, 2, 2)
plt.title("label")
plt.imshow(label[:, :, 60])
plt.show()

# Put data into an iterable dataset
If there is a GPU, we're doing I want to cache the dataset for increased speed... otherwise I am debugging and not concerned with speed

In [ ]:
if device.type == 'cuda':
    train_ds = CacheDataset(
        data=train_files, 
        transform=train_transforms,
        cache_rate=1.0, 
        num_workers=4
    )
    val_ds = CacheDataset(
        data=val_files, 
        transform=val_transforms, 
        cache_rate=1.0, 
        num_workers=4
    )
else:
    train_ds = Dataset(data=train_files, transform=train_transforms)
    val_ds = Dataset(data=val_files, transform=val_transforms)
    
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=2, num_workers=4)

In [ ]:
for batch_data in train_loader:
    inputs, labels = (
        batch_data["image"],
        batch_data["label"],
    )
    break

In [ ]:
print(inputs.shape)
print(labels.shape)

In [ ]:
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
)

In [ ]:
model

In [ ]:
from UNet2D import UNet
model = UNet()

In [ ]:
# Loss functions
loss_function = DiceLoss(to_onehot_y=True, softmax=True)
# # average weight throughout entire label set
# weights = torch.tensor([0.5075946593908832, 33.41786861437203])
# weights = torch.tensor([0.5, 32]) # rounded from above
# loss_function = DiceCELoss(to_onehot_y=True, softmax=True, ce_weight=weights.to(device))
optimizer = torch.optim.Adam(model.parameters(), 1e-2)
dice_metric = DiceMetric(include_background=False, reduction="mean")
inferer = SliceInferer(roi_size=(197, 233), sw_batch_size=1, spatial_dim=4)

In [ ]:
max_epochs = 5
val_interval = 2
best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []
post_pred = Compose([EnsureType(), AsDiscrete(argmax=True, to_onehot=2)])
post_label = Compose([EnsureType(), AsDiscrete(to_onehot=2)])

for epoch in range(max_epochs):
    print("-" * 10)
    print(f"epoch {epoch + 1}/{max_epochs}")
    model.train()
    epoch_loss = 0
    step = 0
    for batch_data in train_loader:
        step += 1
        inputs, labels = (
            batch_data["image"].to(device),
            batch_data["label"].to(device),
        )
        generator = slice_generator(inputs, labels, 32, downscale=False)
        for inputs, labels in generator:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(
            f"{step}/{len(train_ds) // train_loader.batch_size}, "
            f"train_loss: {loss.item():.4f}")
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"epoch {epoch + 1} average loss: {epoch_loss:.4f}")

    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad():
            for val_data in val_loader:
                val_inputs, val_labels = (
                    val_data["image"].to(device),
                    val_data["label"].to(device),
                )
                val_outputs = inferer(val_inputs, model)
                val_outputs = [post_pred(i) for i in decollate_batch(val_outputs)]
                val_labels = [post_label(i) for i in decollate_batch(val_labels)]
                # compute metric for current iteration
                dice_metric(y_pred=val_outputs, y=val_labels)

            # aggregate the final mean dice result
            metric = dice_metric.aggregate().item()
            # reset the status for next validation round
            dice_metric.reset()

            metric_values.append(metric)
            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), os.path.join(
                    training_dir, "best_metric_model.pth"))
                print("saved new best metric model")
            print(
                f"current epoch: {epoch + 1} current mean dice: {metric:.4f}"
                f"\nbest mean dice: {best_metric:.4f} "
                f"at epoch: {best_metric_epoch}"
            )

In [ ]:
print(
    f"train completed, best_metric: {best_metric:.4f} "
    f"at epoch: {best_metric_epoch}")

In [ ]:
plt.figure("train", (12, 6))
plt.subplot(1, 2, 1)
plt.title("Epoch Average Loss")
x = [i + 1 for i in range(len(epoch_loss_values))]
y = epoch_loss_values
plt.xlabel("epoch")
plt.plot(x, y)
plt.subplot(1, 2, 2)
plt.title("Val Mean Dice")
x = [val_interval * (i + 1) for i in range(len(metric_values))]
y = metric_values
plt.xlabel("epoch")
plt.plot(x, y)
plt.savefig('figures.png')
plt.show()

In [ ]:
model.load_state_dict(torch.load(
    os.path.join(training_dir, "best_metric_model.pth")))
model.eval()
with torch.no_grad():
    for i, val_data in enumerate(val_loader):
        roi_size = (160, 160, 160)
        sw_batch_size = 4
        val_outputs = sliding_window_inference(
            val_data["image"].to(device), roi_size, sw_batch_size, model
        )
        # plot the slice [:, :, 80]
        plt.figure("check", (18, 6))
        plt.subplot(1, 3, 1)
        plt.title(f"image {i}")
        plt.imshow(val_data["image"][0, 0, :, :, 80], cmap="gray")
        plt.subplot(1, 3, 2)
        plt.title(f"label {i}")
        plt.imshow(val_data["label"][0, 0, :, :, 80])
        plt.subplot(1, 3, 3)
        plt.title(f"output {i}")
        plt.imshow(torch.argmax(
            val_outputs, dim=1).detach().cpu()[0, :, :, 80])
        plt.savefig('scans.png')
        plt.show()
        if i == 2:
            break
    